# Deepfake Audio Detection Using SVMs — Local Edition
## Run on your laptop (no Google Colab or Drive required)

**Course:** 22MAT220 – Mathematics for Computing III | **Theme A:** Optimization & Machine Learning

---

### Quick Start
1. Install dependencies: `pip install -e .` or `pip install -r requirements.txt`
2. Download ASVspoof 2019 LA → place in `./data/LA/`
3. Run each cell below in order
4. Or use the CLI: `python run_local.py --data-dir ./data --mode quick`


## Step 1 — Install Dependencies


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
print('Dependencies installed. (If this fails, run: pip install -r requirements.txt)')


## Step 2 — Import Libraries


In [ ]:
import os, sys, json, warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import librosa, librosa.display
import sklearn
from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score
from src.svm_from_scratch import LinearSVMScratch, numerical_gradient_check
from src.features import extract_all_features, FeatureConfig
from src.preprocessing import load_audio, normalize_waveform, segment_waveform
from src.calibration import PlattCalibrator
from src.evaluation import compute_metrics
warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'NumPy {np.__version__} | sklearn {sklearn.__version__} | librosa {librosa.__version__}')
# Configure local paths (NO Google Drive needed)
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR,
          os.path.join(RESULTS_DIR,'figures'),
          os.path.join(RESULTS_DIR,'metrics')]:
    os.makedirs(d, exist_ok=True)
print(f'Working directory: {BASE_DIR}')
print(f'Data directory:   {DATA_DIR}')


## Step 3 — Configure Dataset

Place ASVspoof 2019 LA files in:
```
  data/LA/ASVspoof2019_LA_train/flac/
  data/LA/ASVspoof2019_LA_dev/flac/
  data/LA/ASVspoof2019_LA_eval/flac/
  data/LA/ASVspoof2019_LA_cm_protocols/
```
Set `MODE = 'quick'` for 250 samples/class (fast), or `'full'` for all data.


In [ ]:
MODE = 'quick'  # 'quick' or 'full'
SR = 16000
MAX_PER_CLASS = 250 if MODE == 'quick' else None

ASVSPOOF_DIR = os.path.join(DATA_DIR, 'LA')
PROTO_DIR = os.path.join(ASVSPOOF_DIR, 'ASVspoof2019_LA_cm_protocols')

def parse_protocol(path):
    labels = {}
    if not os.path.exists(path): print(f'WARNING: {path} not found'); return labels
    with open(path) as f:
        for line in f:
            p = line.strip().split()
            if len(p)>=5: labels[p[1]] = 0 if p[-1]=='bonafide' else 1
    return labels

def build_list(audio_dir, label_dict, max_cls=None):
    files, labels = [], []
    if not os.path.isdir(audio_dir): print(f'WARNING: {audio_dir} not found'); return files, labels
    rc, fc = 0, 0
    for f in sorted(os.listdir(audio_dir)):
        if not (f.endswith('.flac') or f.endswith('.wav')): continue
        key = f.replace('.flac','').replace('.wav','')
        if key not in label_dict: continue
        lab = label_dict[key]
        if max_cls and ((lab==0 and rc>=max_cls) or (lab==1 and fc>=max_cls)): continue
        files.append(os.path.join(audio_dir, f)); labels.append(lab)
        if lab==0: rc+=1
        else: fc+=1
    return files, labels

train_dict = parse_protocol(os.path.join(PROTO_DIR, 'ASVspoof2019.LA.cm.train.trn.txt'))
dev_dict   = parse_protocol(os.path.join(PROTO_DIR, 'ASVspoof2019.LA.cm.dev.ndx.txt'))
eval_dict  = parse_protocol(os.path.join(PROTO_DIR, 'ASVspoof2019.LA.cm.eval.trl.txt'))

train_files, train_labels = build_list(os.path.join(ASVSPOOF_DIR,'ASVspoof2019_LA_train','flac'), train_dict, MAX_PER_CLASS)
dev_files,   dev_labels   = build_list(os.path.join(ASVSPOOF_DIR,'ASVspoof2019_LA_dev','flac'), dev_dict, MAX_PER_CLASS)
eval_files,  eval_labels  = build_list(os.path.join(ASVSPOOF_DIR,'ASVspoof2019_LA_eval','flac'), eval_dict, MAX_PER_CLASS)

print(f'Train: {len(train_files)} | Dev: {len(dev_files)} | Eval: {len(eval_files)}')
if len(train_files)==0:
    print('No data found. Download ASVspoof 2019 LA and place in data/LA/')


## Step 4 — Extract Features


In [ ]:
cfg = FeatureConfig(sr=SR)
cache_dir = os.path.join(DATA_DIR, 'features_cache')
os.makedirs(cache_dir, exist_ok=True)

def extract_batch(files, labels, name):
    cache = os.path.join(cache_dir, f'{name}.npz')
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        return d['X'], d['y']
    feats, vlabs = [], []
    for i,(f,l) in enumerate(zip(files, labels)):
        if not os.path.exists(f): continue
        try:
            y, _ = load_audio(f, target_sr=SR)
            y = normalize_waveform(y, 'peak')
            feats.append(extract_all_features(y, cfg))
            vlabs.append(l)
        except Exception as e:
            print(f'Skip {f}: {e}')
        if (i+1)%200==0: print(f'  {name}: {i+1}/{len(files)}')
    X = np.array(feats, dtype=np.float32); y = np.array(vlabs)
    np.savez_compressed(cache, X=X, y=y)
    return X, y

X_train, y_train = extract_batch(train_files, train_labels, 'train')
X_dev, y_dev = extract_batch(dev_files, dev_labels, 'dev')
X_eval, y_eval = extract_batch(eval_files, eval_labels, 'eval')
print(f'Feature dim: {X_train.shape[1] if len(X_train)>0 else "N/A"}')


## Step 5 — Train From-Scratch SVM

Mathematical formulation: minimize (1/2)||w||^2 + (C/n) * sum hinge_loss
Subgradient descent with Pegasos learning rate schedule.


In [ ]:
X_fit, X_cal, y_fit, y_cal = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()
X_fit_sc = scaler.fit_transform(imputer.fit_transform(X_fit))
X_dev_sc = scaler.transform(imputer.transform(X_dev))
X_cal_sc = scaler.transform(imputer.transform(X_cal))

svm = LinearSVMScratch(C=1.0, max_epochs=100, batch_size=64, verbose=True)
svm.fit(X_fit_sc, y_fit)
print(f'Training time: {svm.training_time_:.2f}s | Support vectors: {svm.n_support_}')

# Convergence plot
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(svm.obj_history_, color='#00d4ff', lw=2)
axes[0].set_yscale('log'); axes[0].set_title('SVM Optimization Loss')
axes[1].plot(svm.acc_history_, color='#3fb950', lw=2)
axes[1].set_title('Training Accuracy')
plt.tight_layout(); plt.show()

# Gradient check
gc = numerical_gradient_check(svm, X_fit_sc[:200], y_fit[:200])
print(f'Gradient check: max_rel_err={gc["max_relative_error"]:.6f} ({"PASS" if gc["passed"] else "FAIL"})')


## Step 6 — Train sklearn Baselines + Calibration


In [ ]:
pipe = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                 ('scaler', StandardScaler()),
                 ('clf', SVC(kernel='rbf', probability=True))])
gs = GridSearchCV(pipe, {'clf__C':[0.1,1,10], 'clf__gamma':['scale',0.01]},
                  cv=StratifiedKFold(3, shuffle=True, random_state=42),
                  scoring='f1', n_jobs=-1)
gs.fit(X_fit, y_fit)
best_skl = gs.best_estimator_
print(f'Best SVC(RBF): {gs.best_params_} CV F1={gs.best_score_:.4f}')

# Platt calibration for scratch SVM
platt = PlattCalibrator()
platt.fit(svm.decision_function(X_cal_sc), y_cal)
print(f'Platt: A={platt.A:.4f} B={platt.B:.4f}')


## Step 7 — Evaluate


In [ ]:
# Scratch SVM
yp = svm.predict(X_dev_sc)
print(f'Scratch SVM — Dev: acc={accuracy_score(y_dev,yp):.4f} f1={f1_score(y_dev,yp):.4f}')

# sklearn
yp2 = best_skl.predict(X_dev)
print(f'SVC(RBF) — Dev: acc={accuracy_score(y_dev,yp2):.4f} f1={f1_score(y_dev,yp2):.4f}')

# Confusion matrix
if len(X_eval)>0:
    X_eval_sc = scaler.transform(imputer.transform(X_eval))
    yp_eval = svm.predict(X_eval_sc)
    cm = confusion_matrix(y_eval, yp_eval)
    print(f'Scratch SVM — Eval: acc={accuracy_score(y_eval,yp_eval):.4f} f1={f1_score(y_eval,yp_eval):.4f}')
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
    ax.set_title('Confusion Matrix (Eval)'); plt.show()


## Step 8 — Save Model


In [ ]:
import joblib
bundle = {
    'model_type': 'scratch_svm',
    'scratch_w': svm.w_, 'scratch_b': svm.b_, 'scratch_C': svm.C,
    'sklearn_model': best_skl,
    'imputer': imputer, 'scaler': scaler,
    'calibrator': {'A': platt.A, 'B': platt.B},
    'feature_config': cfg.to_dict(),
    'metadata': {'dataset':'ASVspoof 2019 LA', 'mode':MODE},
}
model_path = os.path.join(MODEL_DIR, 'inference_pipeline.joblib')
joblib.dump(bundle, model_path, compress=3)
print(f'Model saved to {model_path}')

# Reload test
b2 = joblib.load(model_path)
print(f'Reload OK: model_type={b2["model_type"]}, w_norm={np.linalg.norm(b2["scratch_w"]):.4f}')


## Step 9 — Launch EchoVerify AI Web App

Run this cell to launch the interactive Gradio website locally.
It will be available at `http://localhost:7860`.


In [ ]:
from src.app import build_app
from src.inference import InferencePipeline
from src.calibration import PlattCalibrator

# Reconstruct pipeline from saved bundle
cal = PlattCalibrator.from_dict(bundle['calibrator'])
from src.svm_from_scratch import LinearSVMScratch
svm_inf = LinearSVMScratch(C=bundle['scratch_C'], verbose=False)
svm_inf.w_ = bundle['scratch_w']; svm_inf.b_ = bundle['scratch_b']; svm_inf.fitted_ = True

pipeline = InferencePipeline(
    model=svm_inf, cfg=cfg, model_name='scratch_svm',
    metadata=bundle['metadata'], scaler=scaler, imputer=imputer, calibrator=cal)

demo = build_app(pipeline)
demo.launch(share=False, debug=True)  # Set share=True for a public link


## Step 10 — Conclusion

The project runs entirely on your local machine. No Google Colab or Drive needed.

**CLI alternatives:**
- Full pipeline: `python run_local.py --data-dir ./data --mode quick`
- Just the app: `python run_app.py`
